<a href="https://colab.research.google.com/github/Shantanu-Jadhav/Repo1/blob/main/ALS_model_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q implicit scipy pandas numpy


In [21]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares



In [22]:
warnings.filterwarnings("ignore")
np.random.seed(42)

# Create a directory to extract the contents of the zip file
os.makedirs("extracted_zip_data", exist_ok=True)
# Unzip data-farmer's cart 3.zip into the 'extracted_zip_data' directory.
# The -o flag is to overwrite existing files without prompting.
!unzip -o "/content/data-farmer's cart 3.zip" -d extracted_zip_data

DATA_PATH = "/content/als_interactions.csv"   # Updated path to the directly available CSV file in /content/
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

REQUIRED_COLS = {"user_id", "product_obj_id", "interaction_value"}
MIN_INTERACTIONS_FOR_SPLIT = 5   # users below this are kept fully in train (cold-start protection)
TEST_SIZE = 0.2
K_VALUES = [5, 10]

BASELINE_PARAMS = {"factors": 32, "regularization": 0.05, "iterations": 20}

Archive:  /content/data-farmer's cart 3.zip
  inflating: extracted_zip_data/data-farmer's cart/banner_clicks_202609051527.csv  
  inflating: extracted_zip_data/__MACOSX/data-farmer's cart/._banner_clicks_202609051527.csv  
  inflating: extracted_zip_data/data-farmer's cart/cart_recommendation_pairs_t_202609051527.csv  
  inflating: extracted_zip_data/__MACOSX/data-farmer's cart/._cart_recommendation_pairs_t_202609051527.csv  
  inflating: extracted_zip_data/data-farmer's cart/admin_notifications_t_202609051527.csv  
  inflating: extracted_zip_data/__MACOSX/data-farmer's cart/._admin_notifications_t_202609051527.csv  
  inflating: extracted_zip_data/data-farmer's cart/handy_orders_t_202609051527.csv  
  inflating: extracted_zip_data/__MACOSX/data-farmer's cart/._handy_orders_t_202609051527.csv  
  inflating: extracted_zip_data/data-farmer's cart/_backup_cart_t_20260827_1523_202609051527.csv  
  inflating: extracted_zip_data/__MACOSX/data-farmer's cart/.__backup_cart_t_20260827_1523_2026

In [24]:
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print(f"'{DATA_PATH}' not found. Please upload als_interactions.csv:")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded.keys()))
    except ImportError:
        print("Not running in Colab and file not found locally — set DATA_PATH manually.")

print("Using DATA_PATH =", DATA_PATH)


Using DATA_PATH = als_interactions_with_product_name (1).csv


In [25]:
def verify_data(path):
    errors, warnings_ = [], []

    if not os.path.exists(path):
        errors.append(f"File not found: {path}")
        return None, errors, warnings_

    try:
        df = pd.read_csv(path)
    except Exception as e:
        errors.append(f"Failed to read CSV: {e}")
        return None, errors, warnings_

    missing_cols = REQUIRED_COLS - set(df.columns)
    if missing_cols:
        errors.append(f"Missing required columns: {missing_cols}")
        return df, errors, warnings_

    null_counts = df[list(REQUIRED_COLS)].isna().sum()
    if null_counts.sum() > 0:
        errors.append(f"Null values found in required columns: {null_counts[null_counts > 0].to_dict()}")

    for col in ["user_id", "product_obj_id"]:
        if not pd.api.types.is_integer_dtype(df[col]):
            try:
                df[col] = df[col].astype(int)
                warnings_.append(f"Column '{col}' was coerced to int.")
            except Exception:
                errors.append(f"Column '{col}' could not be cast to int.")

    if "interaction_value" in df.columns and (df["interaction_value"] < 0).any():
        errors.append("Negative values found in 'interaction_value'.")

    if len(df) == 0:
        errors.append("File is empty (0 rows).")

    dup_count = df.duplicated(subset=["user_id", "product_obj_id"]).sum()
    if dup_count > 0:
        warnings_.append(
            f"{dup_count} duplicate (user_id, product_obj_id) pairs found — "
            f"will be aggregated by summing interaction_value."
        )

    return df, errors, warnings_


raw_df, fatal_errors, warnings_list = verify_data(DATA_PATH)

print("=== Data Verification ===")
if fatal_errors:
    for e in fatal_errors:
        print("ERROR:", e)
    raise SystemExit("Stopping: data verification failed. Fix the errors above before proceeding.")

print("Verification PASSED.")
for w in warnings_list:
    print("WARNING:", w)

print("\nShape:", raw_df.shape)
print("\nDtypes:\n", raw_df.dtypes)
print("\nSummary stats:\n", raw_df.describe())


=== Data Verification ===
Verification PASSED.

Shape: (54934, 4)

Dtypes:
 user_id               int64
product_obj_id        int64
interaction_value     int64
product_name         object
dtype: object

Summary stats:
             user_id  product_obj_id  interaction_value
count  54934.000000    54934.000000       54934.000000
mean    1240.521098      469.785233          10.608476
std      723.344809      105.827460          26.616785
min       27.000000      369.000000           1.000000
25%      667.000000      395.000000           1.000000
50%     1123.000000      437.000000           3.000000
75%     1750.000000      497.000000           9.000000
max     2877.000000      883.000000        1348.000000


In [26]:
df_agg = (
    raw_df.groupby(["user_id", "product_obj_id"], as_index=False)["interaction_value"]
    .sum()
)

user_ids = df_agg["user_id"].unique()
item_ids = df_agg["product_obj_id"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_item = {i: it for it, i in item_to_idx.items()}

df_agg["user_idx"] = df_agg["user_id"].map(user_to_idx)
df_agg["item_idx"] = df_agg["product_obj_id"].map(item_to_idx)

n_users, n_items = len(user_ids), len(item_ids)
print(f"Users: {n_users} | Items: {n_items} | Interactions (post-aggregation): {len(df_agg)}")


Users: 1024 | Items: 332 | Interactions (post-aggregation): 54934


In [27]:
full_matrix = sp.csr_matrix(
    (df_agg["interaction_value"].astype(np.float32), (df_agg["user_idx"], df_agg["item_idx"])),
    shape=(n_users, n_items),
)
print("Full matrix shape:", full_matrix.shape, "| nnz:", full_matrix.nnz,
      f"| density: {full_matrix.nnz / (n_users * n_items):.4%}")


Full matrix shape: (1024, 332) | nnz: 54934 | density: 16.1586%


In [28]:
def train_test_split_implicit(interactions_df, test_size=TEST_SIZE,
                               min_interactions=MIN_INTERACTIONS_FOR_SPLIT, seed=42):
    train_rows, test_rows = [], []
    for _, group in interactions_df.groupby("user_idx"):
        if len(group) < min_interactions:
            train_rows.append(group)
            continue
        group = group.sample(frac=1, random_state=seed)
        n_test = max(1, int(len(group) * test_size))
        test_rows.append(group.iloc[:n_test])
        train_rows.append(group.iloc[n_test:])

    train_df = pd.concat(train_rows, ignore_index=True)
    test_df = pd.concat(test_rows, ignore_index=True) if test_rows else pd.DataFrame(columns=interactions_df.columns)
    return train_df, test_df


train_df, test_df = train_test_split_implicit(df_agg)

train_pairs = set(zip(train_df["user_idx"], train_df["item_idx"]))
test_pairs = set(zip(test_df["user_idx"], test_df["item_idx"]))
leak_count = len(train_pairs & test_pairs)

print(f"Train interactions: {len(train_df)}")
print(f"Test interactions:  {len(test_df)}")
print(f"Users with held-out test interactions: {test_df['user_idx'].nunique()} / {n_users}")
print(f"Leakage check (must be 0): {leak_count}")
assert leak_count == 0, "Data leakage detected between train and test sets!"

train_matrix = sp.csr_matrix(
    (train_df["interaction_value"].astype(np.float32), (train_df["user_idx"], train_df["item_idx"])),
    shape=(n_users, n_items),
)
print("Train matrix shape:", train_matrix.shape, "| nnz:", train_matrix.nnz)


Train interactions: 44358
Test interactions:  10576
Users with held-out test interactions: 990 / 1024
Leakage check (must be 0): 0
Train matrix shape: (1024, 332) | nnz: 44358


In [29]:
def get_recommendations(model, user_items_matrix, user_idx, idx_to_item_map, N=10):
    item_indices, scores = model.recommend(
        user_idx, user_items_matrix[user_idx], N=N, filter_already_liked_items=True
    )
    # Filter out invalid item indices before lookup in idx_to_item_map
    valid_recs = []
    for i, s in zip(item_indices, scores):
        if 0 <= i < len(idx_to_item_map): # Ensure index is within the valid range [0, n_items-1]
            valid_recs.append((idx_to_item_map[i], float(s)))
        else:
            print(f"WARNING: User {user_idx} - Invalid item index {i} returned by model.recommend. Skipping.")
    return valid_recs

# Initialize and train the baseline model
baseline_model = AlternatingLeastSquares(**BASELINE_PARAMS)
baseline_model.fit(train_matrix) # Changed from train_matrix.T to train_matrix

rec_rows = []
for u_idx in range(n_users):
    recs = get_recommendations(baseline_model, train_matrix, u_idx, idx_to_item, N=10)
    for rank, (product_obj_id, score) in enumerate(recs, start=1):
        rec_rows.append({
            "user_id": idx_to_user[u_idx],
            "rank": rank,
            "product_obj_id": product_obj_id,
            "score": score,
        })

rec_df = pd.DataFrame(rec_rows)
rec_df.to_csv(f"{RESULTS_DIR}/top10_recommendations_baseline.csv", index=False)
print(f"Saved {len(rec_df)} recommendation rows -> {RESULTS_DIR}/top10_recommendations_baseline.csv")
rec_df.head(10)

  0%|          | 0/20 [00:00<?, ?it/s]

Saved 10240 recommendation rows -> results/top10_recommendations_baseline.csv


,user_id,rank,product_obj_id,score
0,27,1,424,1.005868
1,27,2,422,0.977704
2,27,3,374,0.971147
3,27,4,387,0.943985
4,27,5,392,0.937731
5,27,6,428,0.936450
6,27,7,425,0.929423
7,27,8,385,0.922174
8,27,9,388,0.900203
9,27,10,441,0.878032


In [30]:
def precision_recall_at_k(model, train_matrix, test_df, k_values=K_VALUES):
    test_by_user = test_df.groupby("user_idx")["item_idx"].apply(set).to_dict()
    max_k = max(k_values)

    results = {f"precision@{k}": [] for k in k_values}
    results.update({f"recall@{k}": [] for k in k_values})

    evaluated_users = 0
    for u_idx, true_items in test_by_user.items():
        if not true_items:
            continue
        item_indices, _ = model.recommend(
            u_idx, train_matrix[u_idx], N=max_k, filter_already_liked_items=True
        )
        evaluated_users += 1
        for k in k_values:
            top_k = set(item_indices[:k])
            hits = len(top_k & true_items)
            results[f"precision@{k}"].append(hits / k)
            results[f"recall@{k}"].append(hits / len(true_items))

    summary = {m: (float(np.mean(v)) if v else 0.0) for m, v in results.items()}
    summary["evaluated_users"] = evaluated_users
    return summary


baseline_metrics = precision_recall_at_k(baseline_model, train_matrix, test_df)
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


Baseline metrics:
  precision@5: 0.4224
  precision@10: 0.3478
  recall@5: 0.2062
  recall@10: 0.3375
  evaluated_users: 990


In [31]:
def catalog_coverage(rec_df, n_items):
    return rec_df["product_obj_id"].nunique() / n_items

coverage = catalog_coverage(rec_df, n_items)

per_user_counts = df_agg.groupby("user_id").size()
per_item_counts = df_agg.groupby("product_obj_id").size()

cold_start_users = int((per_user_counts < MIN_INTERACTIONS_FOR_SPLIT).sum())
cold_start_items = int((per_item_counts < MIN_INTERACTIONS_FOR_SPLIT).sum())

print(f"Catalog coverage (baseline, top-10): {coverage:.2%}")
print(f"Cold-start users (< {MIN_INTERACTIONS_FOR_SPLIT} interactions): "
      f"{cold_start_users} / {n_users} ({cold_start_users/n_users:.1%})")
print(f"Cold-start items (< {MIN_INTERACTIONS_FOR_SPLIT} interactions): "
      f"{cold_start_items} / {n_items} ({cold_start_items/n_items:.1%})")


Catalog coverage (baseline, top-10): 56.93%
Cold-start users (< 5 interactions): 34 / 1024 (3.3%)
Cold-start items (< 5 interactions): 23 / 332 (6.9%)


In [32]:
def train_als(train_matrix, **params):
    model = AlternatingLeastSquares(**params)
    model.fit(train_matrix)
    return model

param_grid = [
    {"factors": 32, "regularization": 0.05, "iterations": 20},  # baseline (for reference)
    {"factors": 64, "regularization": 0.05, "iterations": 20},  # more latent factors
    {"factors": 32, "regularization": 0.10, "iterations": 20},  # stronger regularization
    {"factors": 32, "regularization": 0.05, "iterations": 30},  # more iterations
]

comparison_rows = []
for params in param_grid:
    model = train_als(train_matrix, **params)
    metrics = precision_recall_at_k(model, train_matrix, test_df)
    row = {**params, **metrics}
    comparison_rows.append(row)
    print(params, "->", {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(f"{RESULTS_DIR}/als_param_comparison.csv", index=False)
comparison_df

  0%|          | 0/20 [00:00<?, ?it/s]

{'factors': 32, 'regularization': 0.05, 'iterations': 20} -> {'precision@5': 0.4279, 'precision@10': 0.3531, 'recall@5': 0.2078, 'recall@10': 0.3442, 'evaluated_users': 990}


  0%|          | 0/20 [00:00<?, ?it/s]

{'factors': 64, 'regularization': 0.05, 'iterations': 20} -> {'precision@5': 0.3774, 'precision@10': 0.273, 'recall@5': 0.1892, 'recall@10': 0.2835, 'evaluated_users': 990}


  0%|          | 0/20 [00:00<?, ?it/s]

{'factors': 32, 'regularization': 0.1, 'iterations': 20} -> {'precision@5': 0.4325, 'precision@10': 0.3519, 'recall@5': 0.2131, 'recall@10': 0.3414, 'evaluated_users': 990}


  0%|          | 0/30 [00:00<?, ?it/s]

{'factors': 32, 'regularization': 0.05, 'iterations': 30} -> {'precision@5': 0.4289, 'precision@10': 0.3453, 'recall@5': 0.2133, 'recall@10': 0.3359, 'evaluated_users': 990}


,factors,regularization,iterations,precision@5,precision@10,recall@5,recall@10,evaluated_users
0,32,0.05,20,0.427879,0.353131,0.207770,0.344231,990
1,64,0.05,20,0.377374,0.273030,0.189249,0.283535,990
2,32,0.10,20,0.432525,0.351919,0.213097,0.341374,990
3,32,0.05,30,0.428889,0.345253,0.213292,0.335949,990


In [33]:
best_row = comparison_df.loc[comparison_df["precision@10"].idxmax()]
print("Best configuration (by Precision@10):")
print(best_row)


Best configuration (by Precision@10):
factors             32.000000
regularization       0.050000
iterations          20.000000
precision@5          0.427879
precision@10         0.353131
recall@5             0.207770
recall@10            0.344231
evaluated_users    990.000000
Name: 0, dtype: float64


In [34]:
report_lines = []
report_lines.append("# ALS Baseline Evaluation Report — FarmersKart\n")
report_lines.append(f"Data source: `{DATA_PATH}`\n")
report_lines.append(f"- Users: {n_users}\n- Items: {n_items}\n"
                     f"- Total interactions (post-aggregation): {len(df_agg)}\n"
                     f"- Train interactions: {len(train_df)}\n- Test interactions: {len(test_df)}\n")

report_lines.append("\n## Baseline configuration\n")
report_lines.append(f"`{BASELINE_PARAMS}`\n")

report_lines.append("\n## Baseline metrics\n")
for k, v in baseline_metrics.items():
    report_lines.append(f"- {k}: {v}\n")

report_lines.append("\n## Coverage & cold-start\n")
report_lines.append(f"- Catalog coverage (top-10): {coverage:.2%}\n")
report_lines.append(f"- Cold-start users (< {MIN_INTERACTIONS_FOR_SPLIT} interactions): "
                     f"{cold_start_users} ({cold_start_users/n_users:.1%})\n")
report_lines.append(f"- Cold-start items (< {MIN_INTERACTIONS_FOR_SPLIT} interactions): "
                     f"{cold_start_items} ({cold_start_items/n_items:.1%})\n")

report_lines.append("\n## Parameter comparison\n")
report_lines.append(comparison_df.to_markdown(index=False) + "\n")

report_lines.append("\n## Best configuration (by Precision@10)\n")
report_lines.append(f"`{best_row.to_dict()}`\n")

report_lines.append("\n## Warnings encountered\n")
if warnings_list:
    for w in warnings_list:
        report_lines.append(f"- {w}\n")
else:
    report_lines.append("- None\n")

report_text = "".join(report_lines)
with open(f"{RESULTS_DIR}/als_evaluation_report.md", "w") as f:
    f.write(report_text)

print(f"Report saved -> {RESULTS_DIR}/als_evaluation_report.md")
print(report_text)


Report saved -> results/als_evaluation_report.md
# ALS Baseline Evaluation Report — FarmersKart
Data source: `als_interactions_with_product_name (1).csv`
- Users: 1024
- Items: 332
- Total interactions (post-aggregation): 54934
- Train interactions: 44358
- Test interactions: 10576

## Baseline configuration
`{'factors': 32, 'regularization': 0.05, 'iterations': 20}`

## Baseline metrics
- precision@5: 0.4224242424242424
- precision@10: 0.34777777777777774
- recall@5: 0.20616211649429161
- recall@10: 0.3374614471219283
- evaluated_users: 990

## Coverage & cold-start
- Catalog coverage (top-10): 56.93%
- Cold-start users (< 5 interactions): 34 (3.3%)
- Cold-start items (< 5 interactions): 23 (6.9%)

## Parameter comparison
|   factors |   regularization |   iterations |   precision@5 |   precision@10 |   recall@5 |   recall@10 |   evaluated_users |
|----------:|-----------------:|-------------:|--------------:|---------------:|-----------:|------------:|------------------:|
|        32

In [35]:
import os
import zipfile
import pandas as pd

PRODUCT_CSV_NAME = "product_t_202609051527.csv"
ZIP_INNER_PATH = f"data-farmer's cart/{PRODUCT_CSV_NAME}"

def load_product_table():
    # 1) already extracted / uploaded directly as a CSV
    if os.path.exists(PRODUCT_CSV_NAME):
        return pd.read_csv(PRODUCT_CSV_NAME)

    # 2) look for any zip already in the Colab session
    zip_candidates = [f for f in os.listdir(".") if f.lower().endswith(".zip")]
    for zpath in zip_candidates:
        with zipfile.ZipFile(zpath) as z:
            matches = [n for n in z.namelist() if n.endswith(PRODUCT_CSV_NAME)]
            if matches:
                with z.open(matches[0]) as f:
                    return pd.read_csv(f)

    # 3) nothing found — prompt an upload (either the zip or the CSV directly)
    from google.colab import files
    print(f"Could not find '{PRODUCT_CSV_NAME}'. Upload it, or upload the zip containing it:")
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))

    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(fname) as z:
            matches = [n for n in z.namelist() if n.endswith(PRODUCT_CSV_NAME)]
            if not matches:
                raise FileNotFoundError(f"'{PRODUCT_CSV_NAME}' not found inside {fname}")
            with z.open(matches[0]) as f:
                return pd.read_csv(f)
    else:
        return pd.read_csv(fname)

product_df = load_product_table()
print(f"Loaded {len(product_df)} products.")
product_df.head()

Loaded 353 products.


,id_product,product_name,category_id,sub_category_id,image_url,product_description,product_type,id_hsn,id_gst,product_priority,id_brand,product_rating,active,onDelete,created_At,updated_At
0,368,Tomato,86,108,https://farmerskart.sgp1.digitaloceanspaces.co...,.,New in the category,9.0,8.0,1,23.0,0,1,0,2023-03-18 10:10:09,NaN
1,369,Potato Agara,86,108,https://farmerskart.sgp1.digitaloceanspaces.co...,.,New in the category,9.0,8.0,7,23.0,0,1,0,2023-03-18 10:10:10,NaN
2,370,Potato- Indore,86,108,https://farmerskart.sgp1.digitaloceanspaces.co...,.,.,NaN,NaN,7,23.0,0,1,0,2023-03-18 10:10:10,NaN
3,372,Potato,86,108,https://farmerskart.sgp1.digitaloceanspaces.co...,.,.,NaN,NaN,7,23.0,0,0,0,2023-03-18 10:10:11,NaN
4,373,Onion,86,108,https://farmerskart.sgp1.digitaloceanspaces.co...,.,.,NaN,NaN,8,23.0,0,1,0,2023-03-18 10:10:11,NaN


In [36]:
rec_df_named = rec_df.merge(
    product_df[["id_product", "product_name"]],
    left_on="product_obj_id",
    right_on="id_product",
    how="left",
).drop(columns=["id_product"])

unmatched = rec_df_named["product_name"].isna().sum()
if unmatched:
    print(f"WARNING: {unmatched} recommendation rows had no matching product_name.")

rec_df_named.to_csv(f"{RESULTS_DIR}/top10_recommendations_with_names.csv", index=False)
print(f"Saved -> {RESULTS_DIR}/top10_recommendations_with_names.csv")
rec_df_named.head(15)

Saved -> results/top10_recommendations_with_names.csv


,user_id,rank,product_obj_id,score,product_name
0,27,1,424,1.005868,Chilli Less Spicy
1,27,2,422,0.977704,Ginger
2,27,3,374,0.971147,Onion-Old
3,27,4,387,0.943985,Lauki (400-600gm)
4,27,5,392,0.937731,French beans
5,27,6,428,0.936450,Sprout Mix
6,27,7,425,0.929423,Chilli Spicy
7,27,8,385,0.922174,Beet
8,27,9,388,0.900203,Brinjal small
9,27,10,441,0.878032,Spring Onion


In [38]:
# Interactive user lookup — paste as a new cell after Step 6 (rec_df) and product_df are built

# Build the named recommendation table once (skip if rec_df_named already exists)
if "rec_df_named" not in dir():
    rec_df_named = rec_df.merge(
        product_df[["id_product", "product_name"]],
        left_on="product_obj_id", right_on="id_product", how="left"
    ).drop(columns=["id_product"])

def recommend_for_user(user_id):
    user_recs = rec_df_named[rec_df_named["user_id"] == user_id].sort_values("rank")

    if user_recs.empty:
        print(f"\nUser {user_id} does not exist.\n")
        return

    print(f"\nTop-{len(user_recs)} recommendations for user {user_id}:")
    print(user_recs[["rank", "product_obj_id", "product_name", "score"]].to_string(index=False))
    print()

while True:
    raw_input_value = input("Enter user_id (or 'q' to quit): ").strip()
    if raw_input_value.lower() == "q":
        break
    if not raw_input_value.isdigit():
        print("Please enter a valid numeric user_id.\n")
        continue
    recommend_for_user(int(raw_input_value))

Enter user_id (or 'q' to quit): 27

Top-10 recommendations for user 27:
 rank  product_obj_id      product_name    score
    1             424 Chilli Less Spicy 1.005868
    2             422            Ginger 0.977704
    3             374         Onion-Old 0.971147
    4             387 Lauki (400-600gm) 0.943985
    5             392      French beans 0.937731
    6             428        Sprout Mix 0.936450
    7             425      Chilli Spicy 0.929423
    8             385             Beet  0.922174
    9             388     Brinjal small 0.900203
   10             441     Spring Onion  0.878032

Enter user_id (or 'q' to quit): q


In [40]:
def show_user_interactions(als_df, product_df, user_id):
    """Prints ONLY the products this user actually interacted with (real history)."""
    user_hist = als_df[als_df["user_id"] == user_id]

    if user_hist.empty:
        return False  # signals "user not found" to the caller

    # The raw_df (als_df) already contains 'product_name', so no need for merge here.
    # Just sort the existing dataframe.
    user_hist = user_hist.sort_values("interaction_value", ascending=False)

    print(f"\nUser {user_id} interacted with {len(user_hist)} products:")
    print(user_hist[["product_obj_id", "product_name", "interaction_value"]].to_string(index=False))
    return True


def show_user_recommendations(rec_df_named, user_id):
    """Prints ONLY the model's recommended products for this user (separate from history)."""
    user_recs = rec_df_named[rec_df_named["user_id"] == user_id].sort_values("rank")

    if user_recs.empty:
        print(f"\nNo recommendations found for user {user_id}.")
        return

    print(f"\nTop-{len(user_recs)} recommended products for user {user_id}:")
    print(user_recs[["rank", "product_obj_id", "product_name", "score"]].to_string(index=False))


# --- interactive loop ---
while True:
    raw_input_value = input("\nEnter user_id (or 'q' to quit): ").strip()
    if raw_input_value.lower() == "q":
        break
    if not raw_input_value.isdigit():
        print("Please enter a valid numeric user_id.")
        continue

    user_id = int(raw_input_value)
    found = show_user_interactions(raw_df, product_df, user_id)   # section 1: history
    if not found:
        print(f"\nUser {user_id} does not exist.")
        continue
    show_user_recommendations(rec_df_named, user_id)               # section 2: recommendations


Enter user_id (or 'q' to quit): 27

User 27 interacted with 47 products:
 product_obj_id             product_name  interaction_value
            386               Ladyfinger                 90
            438              Methi Bunch                 54
            410              Red Pumpkin                 52
            429                   Sprout                 51
            421             Lemon Medium                 50
            399                 Drumstic                 50
            378     Cabbage (400-600) gm                 45
            425             Chilli Spicy                 45
            487                   Banana                 42
            382                 Cucumber                 35
            417         Sweet Corn whole                 32
            381   Carrot (Mahabaleshwar)                 32
            437               Corriender                 32
            440              Palak Bunch                 32
            376           

In [42]:
def load_users_table():
    if os.path.exists(USERS_CSV_NAME):
        return pd.read_csv(USERS_CSV_NAME)

    zip_candidates = [f for f in os.listdir(".") if f.lower().endswith(".zip")]
    for zpath in zip_candidates:
        with zipfile.ZipFile(zpath) as z:
            # match the exact file, not any name that happens to end with the same suffix
            matches = [
                n for n in z.namelist()
                if "__MACOSX" not in n
                and n.split("/")[-1] == USERS_CSV_NAME   # exact filename match, not endswith
            ]
            if matches:
                with z.open(matches[0]) as f:
                    return pd.read_csv(f)

    from google.colab import files
    print(f"Could not find '{USERS_CSV_NAME}'. Upload it, or upload the zip containing it:")
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(fname) as z:
            matches = [
                n for n in z.namelist()
                if "__MACOSX" not in n and n.split("/")[-1] == USERS_CSV_NAME
            ]
            with z.open(matches[0]) as f:
                return pd.read_csv(f)
    return pd.read_csv(fname)


users_df = load_users_table()

def get_customer_name(users_df, user_id):
    row = users_df[users_df["id"] == user_id]
    if row.empty:
        return None
    name = row.iloc[0]["user_name"]
    if pd.isna(name):
        name = row.iloc[0]["shop_hotel_name"]  # fallback for society/shop/hotel accounts
    return name


# --- wired into your interactive loop ---
while True:
    raw_input_value = input("\nEnter user_id (or 'q' to quit): ").strip()
    if raw_input_value.lower() == "q":
        break
    if not raw_input_value.isdigit():
        print("Please enter a valid numeric user_id.")
        continue

    user_id = int(raw_input_value)
    customer_name = get_customer_name(users_df, user_id)

    if customer_name is None:
        print(f"\nUser {user_id} does not exist.")
        continue

    print(f"\nCustomer: {customer_name} (user_id {user_id})")
    show_user_interactions(raw_df, product_df, user_id)
    show_user_recommendations(rec_df_named, user_id)


Enter user_id (or 'q' to quit): 27

Customer: Sarika Agarwal (user_id 27)

User 27 interacted with 47 products:
 product_obj_id             product_name  interaction_value
            386               Ladyfinger                 90
            438              Methi Bunch                 54
            410              Red Pumpkin                 52
            429                   Sprout                 51
            421             Lemon Medium                 50
            399                 Drumstic                 50
            378     Cabbage (400-600) gm                 45
            425             Chilli Spicy                 45
            487                   Banana                 42
            382                 Cucumber                 35
            417         Sweet Corn whole                 32
            381   Carrot (Mahabaleshwar)                 32
            437               Corriender                 32
            440              Palak Bunch       